## PyPlus 🐍➕

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="https://media2.giphy.com/media/v1.Y2lkPTc5MGI3NjExMmdlZHd0aXVjb2dtZDU3d2ltZjQ4NnFsZTc4eTlvczYwZ2tjemxyNCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/a5viI92PAF89q/giphy.gif" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code locally</h2>
            <span style="color:#f71;">No C++ compiler? No problem! Copy the generated C++ code and run it directly on an online compiler such as <a href="https://godbolt.org" target="_blank">Compiler Explorer</a> or <a href="https://www.onlinegdb.com" target="_blank">OnlineGDB</a>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="https://media3.giphy.com/media/v1.Y2lkPTc5MGI3NjExcWxhNXFuZ3Vpdmxibm9jdGo4ZnJnbWt4ajlxeTB1bG9pampoamJoaiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/cKPViLWvlFwpVDiQhS/giphy.gif" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, we use free open-source models running locally via Ollama. I have used only two models, you can use any of your choice. Make sure you have installed OLLAMA and desired models.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

d:\Projects\Gear 3\Python to C++\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize OpenAI client using the default API key from environment

openai = OpenAI()

# Define the Ollama API base URL using Ollama's OpenAI-compatible endpoint
ollama_url = "http://localhost:11434/v1"

# Initialize Ollama client
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [3]:
# List of light-weight models.

models = ["llama3.2:3b", "qwen2.5-coder:3b"]

In [4]:
# Import the system info retrieval function from the system_info module
from system_info import retrieve_system_info

# Fetch current system details (e.g., OS, CPU, memory, Python version)
system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-10310U CPU @ 1.70GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (Rev8, Built by MSYS2 project) 15.2.0',
   'g++': 'g++.EXE (Rev8, Built by MSYS2 project) 15.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [ ]:
# Compose a prompt asking the AI to assess C++ compiler setup based on system info
# and provide compile/run commands or installation instructions if needed
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

# Send the prompt to gpt-5-nano and display the response as formatted Markdown
response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

In [5]:
# Compile main.cpp using g++ with C++17 standard and maximum optimizations for native architecture
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "main.cpp", "-o", "main.exe"]

# Execute the compiled binary
run_command = ["main.exe"]

## And now, on with the main task

In [6]:
# System prompt instructing the AI to convert Python code to C++
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

# Build a user prompt that includes system info and compile command for context
def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [7]:
# Build the messages list with system and user roles for the chat completion API
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [8]:
# Write the generated C++ code to main.cpp file
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [9]:
# Convert Python code to C++ using the specified model
def port(model, python):
    # Select the Ollama API client based on the model name
    client = ollama
    
    # Send the conversion request to the model
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    
    # Strip markdown code fences from the response if present
    reply = reply.replace('```cpp','').replace('```','')
    return reply

In [10]:
#Default Python code to be converted, can also be edited on ui.
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [11]:
# Execute a Python code string and capture its stdout output
def run_python(code):
    # Create an isolated global namespace with builtins only
    globals_dict = {"__builtins__": __builtins__}

    # Redirect stdout to a string buffer to capture printed output
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        # Execute the code and retrieve captured output
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        # Return error message if execution fails
        output = f"Error: {e}"
    finally:
        # Restore original stdout regardless of success or failure
        sys.stdout = old_stdout

    return output

In [12]:
# Write, compile, and execute the generated C++ code, returning its output
def compile_and_run(code):
    # Save the C++ code to main.cpp
    write_output(code)
    try:
        # Compile the code and run the resulting executable
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        # Return compiler or runtime error details if the process fails
        return f"An error occurred:\n{e.stderr}"

In [13]:
# Import custom CSS styles for the Gradio interface
from styles import CSS

# Build the Gradio UI layout with a Monochrome theme and custom styling
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"PyPlus 🐍➕") as ui:
    gr.Markdown("## PyPlus 🐍➕")
    
    # Side-by-side code editors for Python input and generated C++ output
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=pi,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label="C++ (generated)",
                value="",
                language="cpp",
                lines=26
            )

    # Control row with run, model selection, convert, and C++ execution buttons
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button("Port to C++", elem_classes=["convert-btn"])
        cpp_run = gr.Button("Run C++", elem_classes=["run-btn", "cpp"])

    # Side-by-side output areas for Python and C++ execution results
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label= "C++ result", lines=8, elem_classes=["cpp-out"])

    # Wire buttons to their respective backend functions
    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

# Launch the UI and open it automatically in the browser
ui.launch(inbrowser=True)

C:\Users\dell\AppData\Local\Temp\ipykernel_10496\961279587.py:5: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"PyPlus 🐍➕") as ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
